In [1]:
# Cell 1: Setup, Models, and DB Connection
import os

# 1. GPU Setup (CRITICAL: These two lines must be at the very top)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # Align PyTorch numbering with nvtop
os.environ["CUDA_VISIBLE_DEVICES"] = "3"        # Lock to physical GPU 2
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"

# 2. NOW it is safe to import the heavy ML libraries
import re
import json
import torch
import chromadb
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# Verify PyTorch setup
print(f"Torch: {torch.__version__}")
if torch.cuda.is_available():
    # This should now say "Device Count: 1" and run specifically on physical GPU 2
    print(f"✅ GPU Locked To: {torch.cuda.get_device_name(0)}")
    print(f"✅ Visible GPUs: {torch.cuda.device_count()}")
else:
    print("❌ No GPU detected.")

# 3. Connect to existing ChromaDB
print("🔗 Connecting to ChromaDB...")
client = chromadb.PersistentClient(path="poetry_db")
collection = client.get_collection(name="hindwi_poems") 

# 4. Load Embedder
print("⏳ Loading Embedder...")
embedder = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')

# 5. Load Qwen LLM
print("⏳ Loading Qwen 2.5 (14B)...")
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda", low_cpu_mem_usage=True
)
model.eval()
print("✅ Ready for experiments!")

/home/literature/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.9.1+cu128
✅ GPU Locked To: NVIDIA RTX A6000
✅ Visible GPUs: 1
🔗 Connecting to ChromaDB...
⏳ Loading Embedder...
⏳ Loading Qwen 2.5 (14B)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 8/8 [00:05<00:00,  1.43it/s]


✅ Ready for experiments!


In [2]:
# Cell 2: PoetryExperimenter Class
class PoetryExperimenter:
    def __init__(self, model, tokenizer, collection, embedder):
        self.model = model
        self.tokenizer = tokenizer
        self.collection = collection
        self.embedder = embedder
        self.results_log = []

    def _generate(self, system_prompt, user_prompt, temperature=0.7, max_tokens=1024):
        """Helper for standard generation."""
        
        # Add a bulletproof language constraint to whatever system prompt is passed
        strict_system = (
            system_prompt + 
            " STRICT RULE: You must respond ONLY in pure Hindi (Devanagari script). "
            "Do NOT output any Chinese, English, or AI conversational filler."
        )

        messages = [
            {"role": "system", "content": strict_system},
            {"role": "user", "content": user_prompt}
        ]
        inputs = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                inputs, max_new_tokens=max_tokens, temperature=temperature, do_sample=True, repetition_penalty=1.1
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

    def find_best_poet(self, topic):
        """Queries ChromaDB to find the poet who has written the most semantically similar poems."""
        try:
            query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
            results = self.collection.query(
                query_embeddings=query_vector,
                n_results=1
            )
            if results['metadatas'] and results['metadatas'][0]:
                return results['metadatas'][0][0]['poet_slug']
        except Exception as e:
            print(f"⚠️ Error finding best poet: {e}")
        return "ramdhari-singh-dinkar" # Fallback

    # 🔹 1. Zero-Shot
    def zero_shot(self, topic):
        return self._generate("आप एक उत्कृष्ट हिंदी कवि हैं।", f"विषय: '{topic}' पर एक कविता लिखें।")

    def few_shot(self, topic, style="ramdhari-singh-dinkar"):
        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
        res = self.collection.query(query_embeddings=query_vector, n_results=2, where={"poet_slug": style})
        
        examples_text = ""
        if res['documents'] and res['documents'][0]:
            for i, text in enumerate(res['documents'][0]):
                example_lines = "\n".join(text.split("\n")[:6]).strip()
                examples_text += f"उदाहरण {i+1}:\n{example_lines}\n\n"
                
        sys = "आप एक उत्कृष्ट हिंदी कवि हैं। नीचे दिए गए उदाहरणों की शैली और संरचना को समझें।"
        user = (
            f"{examples_text}"
            f"अब, विषय: '{topic}' पर एक बिल्कुल नई और मौलिक (original) कविता लिखें।\n"
            f"⚠️ चेतावनी: ऊपर दिए गए उदाहरणों की केवल शैली (style) का पालन करें, उनकी पंक्तियों या शब्दों को अपनी कविता में बिल्कुल न दोहराएं।"
        )
        return self._generate(sys, user)

    def rag_style_conditioned(self, topic, style="ramdhari-singh-dinkar"):
        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
        res = self.collection.query(query_embeddings=query_vector, n_results=2, where={"poet_slug": style})
        
        context = ""
        if res['documents'] and res['documents'][0]:
            for i, text in enumerate(res['documents'][0]):
                context += f"संदर्भ {i+1}:\n{text[:300]}...\n"
                
        user = (
            f"इस शैली का अनुकरण करें:\n{context}\n\n"
            f"विषय: '{topic}' पर एक नई कविता लिखें।\n"
            f"⚠️ महत्वपूर्ण नियम: संदर्भ से सीधे पंक्तियाँ कॉपी न करें। केवल उसी अंदाज़ (tone) में नए शब्द लिखें।"
        )
        return self._generate("आप एक विशेषज्ञ कवि हैं जो मौलिक रचनाएँ करते हैं।", user)

    # 🔹 4. Chain-of-Thought & 5. Plan-Then-Generate
    def plan_then_generate(self, topic):
        sys = "आप एक कवि और आलोचक हैं। पहले कविता की योजना बनाएं, फिर कविता लिखें।"
        user = (
            f"विषय: '{topic}'\n"
            "1. भाव (Mood) तय करें।\n"
            "2. 5 मुख्य शब्द (Vocabulary) चुनें।\n"
            "3. अलंकार (Metaphor) सोचें।\n"
            "4. अंत में 'कविता:' शीर्षक के साथ कविता लिखें।"
        )
        return self._generate(sys, user)

    # 🔹 6. Self-Critique Loop
    def self_critique(self, topic):
        draft = self.zero_shot(topic)
        critique_prompt = f"इस कविता की आलोचना करें और 2 कमियां निकालें (लय या शब्द चयन):\n{draft}"
        critique = self._generate("आप एक कठोर आलोचक हैं।", critique_prompt)
        
        refine_prompt = f"मूल कविता:\n{draft}\n\nआलोचना:\n{critique}\n\nआलोचना को ध्यान में रखते हुए एक श्रेष्ठ संस्करण लिखें।"
        final = self._generate("आप एक मास्टर कवि हैं जो अपनी गलतियों को सुधारता है।", refine_prompt)
        return f"--- DRAFT ---\n{draft}\n\n--- CRITIQUE ---\n{critique}\n\n--- FINAL ---\n{final}"

    # 🔹 7. Constraint-Based
    def constraint_based(self, topic):
        user = (
            f"विषय: '{topic}'\n"
            "नियम:\n"
            "1. कविता में ठीक 4 पंक्तियां (lines) होनी चाहिए।\n"
            "2. 'आसमान' शब्द का प्रयोग वर्जित है।\n"
            "3. अंतिम पंक्ति 'कहानी' शब्द पर खत्म होनी चाहिए।"
        )
        return self._generate("आप नियमों का सख्ती से पालन करने वाले कवि हैं।", user)

    # 🔹 8. Temperature Experiments
    def temp_experiment(self, topic):
        low_temp = self._generate("आप एक कवि हैं।", f"विषय: '{topic}'", temperature=0.2)
        high_temp = self._generate("आप एक कवि हैं।", f"विषय: '{topic}'", temperature=1.2)
        return f"--- TEMP 0.2 (Predictable) ---\n{low_temp}\n\n--- TEMP 1.2 (Creative/Chaotic) ---\n{high_temp}"

    # 🔹 9. Persona-Based
    def persona_based(self, topic):
        sys = "आप 19वीं सदी के एक उदास, दार्शनिक कवि हैं जो पुरानी हिंदी (तद्भव/तत्सम बहुल) में लिखते हैं।"
        return self._generate(sys, f"इस विषय पर अपने विचार प्रकट करें: '{topic}'")

    # 🔹 11. Prompt Engineering Variants
    def prompt_variants(self, topic):
        variant_a = self._generate("कवि बनो।", f"{topic} पर लिखो।")
        variant_b = self._generate(
            "आप साहित्य अकादमी पुरस्कार विजेता हैं। आपकी भाषा हृदय को छू लेने वाली और प्रतीकात्मक है।", 
            f"कृपया '{topic}' विषय पर एक मर्मस्पर्शी रचना प्रस्तुत करें।"
        )
        return f"--- BASIC PROMPT ---\n{variant_a}\n\n--- ENGINEERED PROMPT ---\n{variant_b}"

    # 🔹 12. Multi-Agent Poetry (Model talks to itself)
    def multi_agent(self, topic):
        sys_a = "आप 'कवि A' हैं। आप बहुत ही शांत और प्रकृति-प्रेमी हैं। विषय पर केवल पहली 4 पंक्तियां (Stanza 1) लिखें।"
        stanza_1 = self._generate(sys_a, f"विषय: '{topic}'")
        
        sys_b = "आप 'कवि B' हैं। आपका स्वभाव उग्र और क्रांतिकारी है। 'कवि A' की कविता को आगे बढ़ाते हुए अगली 4 पंक्तियां (Stanza 2) लिखें।"
        stanza_2 = self._generate(sys_b, f"कवि A ने यह लिखा है:\n{stanza_1}\n\nअब आप इसे अपने विद्रोही अंदाज में पूरा करें।")
        
        return f"--- STANZ 1 (Calm Agent) ---\n{stanza_1}\n\n--- STANZA 2 (Fiery Agent) ---\n{stanza_2}"

    # 🔹 13. Automatic Evaluation (Custom Port & 5-Run)
    def auto_eval(self, topic, generated_poem, poet_name, reference_poems, num_evals=5):
        import json
        import re
        import statistics
        from ollama import Client # Import the custom client

        # Point exactly to the custom port we just opened
        ollama_client = Client(host='http://127.0.0.1:11450')

        system_instruction = (
            "You are an expert Hindi literary critic. "
            "Output ONLY a valid JSON object. No explanations, no markdown, no conversational text."
        )
        
        evaluation_prompt = (
            f"Topic: {topic}\n"
            f"Target Poet Style: {poet_name}\n\n"
            f"--- REFERENCE POEMS BY {poet_name} ---\n"
            f"{reference_poems}\n\n"
            f"--- GENERATED POEM TO EVALUATE ---\n"
            f"{generated_poem}\n\n"
            "Evaluate the generated poem on a scale of 1 to 10. "
            "Return EXACTLY this JSON format and nothing else:\n"
            '{"fluency": 0, "coherence": 0, "relevance": 0, "creativity": 0, "style_similarity": 0}'
        )

        final_scores = {"Llama-3.1": {}, "Gemma-2": {}}
        metrics = ['fluency', 'coherence', 'relevance', 'creativity', 'style_similarity']

        def clean_json(text):
            text = text.strip()
            match = re.search(r'\{.*\}', text, re.DOTALL)
            return match.group(0) if match else text

        for model_name, dict_key in [('llama3.1', 'Llama-3.1'), ('gemma2', 'Gemma-2')]:
            raw_scores = {m: [] for m in metrics}
            
            print(f"\n🔍 DEBUG: Running {dict_key} evaluation...")
            
            for i in range(num_evals):
                try:
                    # Use our custom client instead of the default ollama.chat
                    response = ollama_client.chat(model=model_name, messages=[
                        {'role': 'system', 'content': system_instruction},
                        {'role': 'user', 'content': evaluation_prompt}
                    ], options={'temperature': 0.7})
                    
                    raw_text = response['message']['content']
                    clean_text = clean_json(raw_text)
                    parsed = json.loads(clean_text)
                    
                    for m in metrics:
                        if m in parsed:
                            raw_scores[m].append(float(parsed[m]))
                            
                except Exception as e:
                    print(f"❌ DEBUG ERROR on {dict_key} (Attempt {i+1}): {e}")
                    if 'response' in locals():
                        print(f"   Model Output was: {response.get('message', {}).get('content', 'Nothing')}")
            
            # Aggregation logic
            aggregated = {}
            for m in metrics:
                vals = raw_scores[m]
                if len(vals) > 1:
                    aggregated[m] = {
                        "mean": round(statistics.mean(vals), 2),
                        "std_dev": round(statistics.stdev(vals), 2)
                    }
                elif len(vals) == 1:
                    aggregated[m] = {"mean": round(vals[0], 2), "std_dev": 0.0}
                else:
                    aggregated[m] = {"error": "Failed all 5 attempts"}
            
            final_scores[dict_key] = aggregated

        return json.dumps(final_scores, indent=2, ensure_ascii=False)

    # 🔹 The Execution Engine (Updated for Dual-Judge)
    def run_all(self, topics):
        experiments = {
            "Zero-Shot": self.zero_shot,
            "Few-Shot": self.few_shot,
            "RAG Style (Best Match)": self.rag_style_conditioned,
            "Plan-Then-Generate": self.plan_then_generate,
            "Self-Critique": self.self_critique,
            "Constraints": self.constraint_based,
            "Temperature (0.2 vs 1.2)": self.temp_experiment,
            "Persona (19th Century)": self.persona_based,
            "Prompt Variants": self.prompt_variants,
            "Multi-Agent": self.multi_agent
        }

        output_dir = "Qwen_Output"
        os.makedirs(output_dir, exist_ok=True)
        tqdm.write(f"📂 Created main folder: '{output_dir}/'")
        tqdm.write("🚀 Starting Massive Experiment Suite...")
        
        for topic in tqdm(topics, desc="Processing Topics"):
            tqdm.write(f"\n{'='*50}\n🌟 TOPIC: {topic}\n{'='*50}")
            
            best_poet_slug = self.find_best_poet(topic)
            tqdm.write(f"   🎯 Best Poet Found: {best_poet_slug}")
            
            safe_filename = re.sub(r'[^\w\s-]', '', topic).strip().replace(' ', '_')
            file_path = os.path.join(output_dir, f"{safe_filename}.txt")
            
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(f"🌟 TOPIC: {topic}\n")
                f.write(f"🎯 BEST POET MATCH: '{best_poet_slug}'\n")
                f.write(f"{'='*50}\n\n")
                
                for exp_name, exp_func in experiments.items():
                    tqdm.write(f"   🧪 Running: {exp_name}...")
                    f.write(f"🧪 EXPERIMENT: {exp_name}\n")
                    f.write(f"{'-'*50}\n")
                    
                    try:
                        # 1. Fetch Reference Context for the Judges
                        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
                        res = self.collection.query(query_embeddings=query_vector, n_results=3, where={"poet_slug": best_poet_slug})
                        
                        reference_context = ""
                        if res['documents'] and res['documents'][0]:
                            for i, text in enumerate(res['documents'][0]):
                                reference_context += f"Reference {i+1}:\n{text[:300]}...\n"

                        # 2. Generate Poem (Using Qwen 14B)
                        if exp_name == "RAG Style (Best Match)":
                            poem = exp_func(topic, style=best_poet_slug)
                        else:
                            poem = exp_func(topic)
                        
                        # 3. Evaluate Poem (Using Llama 3.1 & Gemma 2 via Ollama)
                        eval_scores_json = self.auto_eval(
                            topic=topic, 
                            generated_poem=poem, 
                            poet_name=best_poet_slug, 
                            reference_poems=reference_context
                        )
                        
                        # 4. Save to File
                        f.write("📜 GENERATED POEM:\n")
                        f.write(poem + "\n\n")
                        f.write("🧠 DUAL-AI EVALUATION (JSON):\n")
                        f.write(eval_scores_json + "\n\n")
                        f.write(f"{'='*50}\n\n")
                        
                    except Exception as e:
                        tqdm.write(f"      ❌ FAILED! Error in {exp_name}: {e}")
                        f.write(f"❌ ERROR GENERATING POEM: {str(e)}\n\n")
                        f.write(f"{'='*50}\n\n")
            
            tqdm.write(f"   💾 Saved results to: {file_path}")

        tqdm.write(f"\n✅ All experiments complete!")

In [3]:
# Cell 3: Run the Massive Experiment Suite
TEST_TOPICS = [
    "🌿 प्रकृति (Nature): बारिश की पहली बूंद",
    "❤️ भावनात्मक (Emotional): अधूरी मोहब्बत",
    "🌍 सामाजिक (Social): नारी शक्ति",
    "🧠 दार्शनिक (Philosophical): समय का चक्र",
    "🎭 रचनात्मक / अनोखा (Creative): टूटी हुई घड़ी की कहानी",
    "🌅 आशावादी (Optimistic): ख्वाबों का आसमान",
    "😔 उदास (Sad): सूनी राहें",
    "🌾 नॉस्टैल्जिक (Nostalgic): मिट्टी की खुशबू",
    "💔 दर्दभरा (Painful): चुप्पी का बोझ",
    "✨ प्रेरणादायक (Inspirational): उम्मीद की किरण",
    "🏡 स्मृतिपूर्ण (Reminiscent): बचपन की गलियाँ",
    "🌆 अकेलापन (Lonely): अजनबी शहर",
    "❤️ रोमांटिक (Romantic): दिल की दस्तक",
    "🤝 भावुक (Emotional): रिश्तों की डोर",
    "⏳ दार्शनिक (Reflective): वक्त की रेत",
    "🕊️ उत्साहपूर्ण (Energetic): सपनों की उड़ान",
    "🎈 निराशाजनक (Hopeless): टूटी हुई पतंग",
    "🪞 गंभीर (Serious): सच का आईना",
    "📜 विरहपूर्ण (Separation): आख़िरी ख़त",
    "🌄 सकारात्मक (Positive): नई सुबह"
]

# Initialize and Run
experimenter = PoetryExperimenter(model, tokenizer, collection, embedder)
experimenter.run_all(TEST_TOPICS)

📂 Created main folder: 'Qwen_Output/'
🚀 Starting Massive Experiment Suite...


Processing Topics:   0%|          | 0/20 [00:00<?, ?it/s]


🌟 TOPIC: 🌿 प्रकृति (Nature): बारिश की पहली बूंद


Processing Topics:   0%|          | 0/20 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


   🎯 Best Poet Found: gopalkrishna-kaul
   🧪 Running: Zero-Shot...

🔍 DEBUG: Running Llama-3.1 evaluation...

🔍 DEBUG: Running Gemma-2 evaluation...


...

   🧪 Running: Prompt Variants...

🔍 DEBUG: Running Llama-3.1 evaluation...

🔍 DEBUG: Running Gemma-2 evaluation...


Processing Topics:  95%|█████████▌| 19/20 [2:45:13<08:51, 531.73s/it]

   🧪 Running: Multi-Agent...

🔍 DEBUG: Running Llama-3.1 evaluation...

🔍 DEBUG: Running Gemma-2 evaluation...


Processing Topics: 100%|██████████| 20/20 [2:45:37<00:00, 496.85s/it]

   💾 Saved results to: Qwen_Output/सकरतमक_Positive_नई_सबह.txt

✅ All experiments complete!
